# GroupGen Study Analysis — Research Questions

This notebook analyzes the **pilot** survey data (especially CRPY) in the shape of the updated study design.

## Research questions

### RQ1 — GroupGen vs teacher grouping
**Independent variable:** Grouping method (`GroupGen` vs `teacher` control).

**Dependent variables:**
1. **Feature trajectories** — academic self-efficacy, classroom engagement, and study habits from Get-Features / Check-in waves
2. **Collaboration quality** — group satisfaction (primary), plus belonging, psychological safety, and team learning from reflection surveys

**Hypothesis (directional):** Students in GroupGen teams report higher collaboration satisfaction (and lower friction) than students in teacher-formed teams. Feature levels over the semester are compared across the same two arms.

### RQ2 — What else matters inside teams?
**Primary predictor (IV of interest):** Within-team **feature similarity** (how close teammates are on the Get-Features profile).

**Covariates:** Age variance within team, demographic isolation (solo gender/ethnicity), group size.

**Dependent variable:** Collaboration / project satisfaction.

**Hypothesis:** Feature similarity remains a significant predictor of satisfaction after controlling for those composition factors.

> **Important:** RQ2's IV is **feature similarity**, not grouping method. Grouping method belongs to RQ1.

## How to read the pilot results
CRPY students experienced conditions **in sequence** (student-choice → teacher → GroupGen), not as parallel randomized arms. Treat contrasts as **exploratory / design-prep**, not confirmatory causal tests.

| Notebook section | What it answers |
|---|---|
| Setup | Paths, packages, CRPY data check |
| Design readiness | What today's data can / cannot support |
| RQ1a | Feature level change across available waves |
| RQ1b | Collaboration outcomes for teacher vs GroupGen |
| RQ2 | Composition regression path (blocked until team roster exists) |
| Instruments | Scale reliability (supports both RQs) |
| Next semester | Exact fields to collect |

**Data:** `docs/Inclass_data/<COURSE>/` · **Outputs:** `docs/output/tables/` and `docs/output/figures/`


## Setup

Run once per session. Uses your local `docs/` folder (recommended) or Colab clone paths.


In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Image
from notebook_bootstrap import init_session
import rq_study_analysis as rq

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 100)

# Set True once to wipe docs/output/ before a fresh run.
CLEAR_OUTPUTS = False

ga, DATA_DIR, setup_info = init_session()
if CLEAR_OUTPUTS:
    removed = ga.clear_outputs()
    print(f"Cleared {removed} old output file(s).")

# Ensure roster template exists for RQ2 planning.
template_path = rq.ensure_roster_template()
print(f"RQ2 roster template: {template_path}")
print("Setup complete.")


## Design readiness

Checklist against the **new** RQs using whatever is currently on disk.

Green-path items can be analyzed now. `blocked` items need next-semester data collection (especially a **team roster** with `student_key`, `team_id`, `condition`).


In [ ]:
results = rq.run_rq_study(ga, course="CRPY", save=True, display=False)

print("Design readiness")
display(results.design_readiness)

print("\nSurvey inventory (condition tags applied to outcome waves)")
display(results.inventory)


---
# RQ1a — Feature trajectories

**Question:** How do student feature levels change across the semester, and (in the full study) do GroupGen and teacher arms differ?

**Pilot instruments**
- Check-in wave → `group_zero_baseline` (feature scores)
- Get-Features wave → `post_regroup_features` (feature scores + demographics)

**Next-semester mapping**
- Get-Features at start = **T0 baseline** (also feeds GroupGen clustering)
- Check-ins during semester = **T1+ follow-ups**
- Compare trajectories by **condition** (`groupgen` vs `teacher`)

The cells below reuse the pilot's two available feature waves to practice the exact paired analysis you will run on both arms.


In [ ]:
print("Paired feature change (all baseline scales)")
if results.feature_change.empty:
    print("No paired feature data found.")
else:
    display(results.feature_change.head(15))
    print("\nPaired tests by scale")
    display(results.feature_change_tests)

fig = ga.OUT_FIGURES / "rq1_feature_trajectories.png"
if not fig.exists():
    rq.make_feature_trajectory_figure(results.feature_change)
if fig.exists():
    display(Image(filename=str(fig)))


### How to interpret RQ1a (pilot)

- Positive `mean_change` = scores rose from the earlier wave to the later wave.
- Non-significant *p*-values with small *n* are expected in the pilot — use means/SDs for power planning.
- Parallel-arm comparison (`groupgen` vs `teacher`) requires next semester's design; this section validates that feature scoring + pairing works.


---
# RQ1b — Collaboration outcomes by grouping condition

**Question:** Do collaboration satisfaction (and related outcomes) differ under GroupGen vs teacher grouping?

**Primary DV:** `group_satisfaction`  
**Secondary DVs:** `group_belonging`, `psychological_safety`, `team_learning`  
*(Psychological safety reverse-scored items are a partial stand-in for "team friction" until a dedicated friction scale is added.)*

**Pilot contrast:** Teacher-Groups reflection vs GroupGen reflection (paired by student name).  
**Caveat:** Same students, sequential weeks — order/time confounds exist.


In [ ]:
print("Outcome means by condition (pilot waves)")
display(results.outcomes_by_condition)

print("\nFocus arms for the planned study: teacher vs groupgen")
focus = results.outcomes_by_condition[
    results.outcomes_by_condition["condition"].isin(["teacher", "groupgen"])
].copy()
display(focus.pivot_table(index="scale", columns="condition", values="mean"))

print("\nPaired change: teacher week -> GroupGen week")
display(results.teacher_vs_groupgen_summary)

for name in ["rq1_outcomes_by_condition.png", "rq1_teacher_vs_groupgen.png"]:
    path = ga.OUT_FIGURES / name
    if not path.exists() and name == "rq1_outcomes_by_condition.png":
        rq.make_outcomes_by_condition_figure(results.outcomes_by_condition)
    if not path.exists() and name == "rq1_teacher_vs_groupgen.png":
        rq.make_teacher_vs_groupgen_figure(results.teacher_vs_groupgen_summary)
    if path.exists():
        display(Image(filename=str(path)))


### How to interpret RQ1b (pilot)

- Compare `teacher` vs `groupgen` means on **group_satisfaction** first (primary DV).
- `mean_change` in the paired table is GroupGen minus teacher for the same students.
- For the confirmatory study, each student should contribute **one** condition outcome (parallel arms), not two sequential reflections.


---
# RQ2 — Feature similarity vs composition covariates

**Question:** Aside from feature-similarity matching, which contextual/demographic team factors relate to satisfaction?

| Role | Variable |
|---|---|
| **IV (primary predictor)** | Within-team feature similarity |
| Covariates | Age variance, demographic isolation, group size |
| DV | Collaboration satisfaction |
| Sample | Preferably GroupGen teams (or both arms with method as an extra factor) |

**Model target:**  
`satisfaction ~ feature_similarity + age_variance + isolation + group_size`

This section is **roster-gated**. Without `team_id` on a roster, team-level covariates cannot be computed.


In [ ]:
print("RQ2 status")
for key, value in results.rq2_status.items():
    print(f"  {key}: {value}")

roster_path = rq.find_roster_path()
print(f"\nRoster file used: {roster_path or '(none found)'}")
print(f"Template to copy/fill: {rq.ROSTER_TEMPLATE_PATH}")

if results.rq2_status.get("ready_for_regression"):
    print("\nTeam composition covariates")
    display(results.team_composition)
    print("\nStudent-level model table (preview)")
    display(results.rq2_model_ready.head(20))
    if not results.rq2_model_ready.empty:
        cols = [
            c
            for c in [
                "satisfaction",
                "feature_similarity",
                "age_variance",
                "pct_gender_isolated",
                "pct_ethnicity_isolated",
                "group_size",
            ]
            if c in results.rq2_model_ready.columns
        ]
        print("\nExploratory correlations")
        display(results.rq2_model_ready[cols].corr(numeric_only=True).round(3))
else:
    print("\nRQ2 analysis is blocked until a team roster exists.")
    print("Create docs/Inclass_data/team_rosters.csv with at least:")
    print("  student_key, team_id, condition")
    print("Optional but needed for full covariates: group_size, age")
    print("Then re-run Setup + Design readiness + this section.")
    display(pd.read_csv(rq.ROSTER_TEMPLATE_PATH))


### What to collect for RQ2 next semester

1. Stable **student_key** on every survey (name and/or email)
2. A roster export: `student_key`, `team_id`, `condition` (`groupgen`|`teacher`), `group_size`
3. **Age** (intake or roster) for within-team age variance
4. Gender / ethnicity on intake (already on Get-Features) for isolation flags
5. Outcome survey still needs student ID so satisfaction can join to the roster


---
# Instrument quality

Reliability and subscale means for the CRPY waves used above. Scales with Cronbach's α ≥ .70 are generally acceptable for group comparisons.


In [ ]:
rel = results.reliability.copy()
display(rel.sort_values(["kind", "scale", "survey"]))

pooled = ga.pooled_scale_summary(rel)
print("Pooled summary (CRPY waves in this run)")
display(pooled)

equity = ga.demographic_equity(results.surveys)
if not equity.empty:
    print("Baseline constructs by demographic (where available)")
    display(equity)
else:
    print("No demographic equity table (demographics missing on scored baseline waves).")


---
# Next-semester checklist

Use this as the data contract for the confirmatory study.

### Design
- [ ] Parallel arms: **GroupGen** vs **teacher** (same course/assignment window)
- [ ] Same Get-Features form at T0 for both arms
- [ ] Same Check-in form at T1(+) for both arms
- [ ] One reflection/outcome survey per student under their assigned condition

### Identifiers & roster
- [ ] Student ID on every form
- [ ] `team_rosters.csv` with `student_key`, `team_id`, `condition`, `group_size`
- [ ] Age collected (intake or roster)

### Analysis entry points in this repo
- Notebook: this file (`GG_analysis.ipynb`)
- Module: `rq_study_analysis.py` (`run_rq_study`)
- Roster template: `Inclass_data/team_roster_template.csv`

When the roster exists, re-run **Setup** → **Design readiness** → **RQ2**; the composition tables and exploratory correlations will populate automatically.


In [ ]:
# One-shot refresh: recompute + save all RQ outputs (with inline display)
results = rq.run_rq_study(ga, course="CRPY", save=True, display=True)
print("Done.")
